# Ejemplo de uso: ajuste conjunto edad + M1 + M2

Este notebook carga las funciones de `isochrone_binary_fit.ipynb` y las
aplica primero a un caso **sintético** (para que puedas correr todo el
notebook de inmediato y ver que funciona), y después muestra dónde
conectar tus datos reales (TIC + BHAC15/PARSEC).


## 0. Cargar la librería

In [ ]:
%run "isochrone_binary_fit.ipynb"

## 1. Caso sintético (sanity check)

Genero una malla de isocronas de juguete con una edad y masas conocidas, agrego ruido observacional, y verifico que el pipeline recupera los valores verdaderos dentro de la incertidumbre.

In [ ]:
import numpy as np
import pandas as pd

logAges = np.linspace(np.log10(1e6), np.log10(200e6), 25)
masses = np.linspace(0.05, 1.2, 40)
rows = []
for la in logAges:
    age_myr = 10**la / 1e6
    for m in masses:
        R = 1.2 * m**0.9 * (age_myr / 10)**(-0.2)      # contrae con el tiempo (juguete)
        Teff = 3000 + 3000 * m**0.6 + 50 * np.log10(age_myr)
        L = R**2 * (Teff / 5772)**4
        rows.append((la, m, Teff, np.log10(L), R))

toy_grid = pd.DataFrame(rows, columns=["logAge", "mass", "Teff", "logL", "radius"])
toy_interp = IsochroneInterpolator(toy_grid, "TOY")

true_age_myr, true_M1, true_M2 = 30.0, 0.9, 0.4
la_true = np.log10(true_age_myr * 1e6)

obs_toy = BinaryObservables(
    Teff1=float(toy_interp.teff(la_true, true_M1)) + 20, sigma_Teff1=50,
    R1=float(toy_interp.radius(la_true, true_M1)) + 0.02, sigma_R1=0.05,
    R2=float(toy_interp.radius(la_true, true_M2)) - 0.01, sigma_R2=0.05,
)
obs_toy

In [ ]:
result_toy = run_grid_search(obs_toy, toy_interp, n_age=80, n_M1=80)
samples_toy = run_posterior_sampling(result_toy, n_samples=5000, random_state=1)
summary_toy = summarize_posterior(samples_toy)
table_toy = build_results_table(obs_toy, result_toy, summary_toy, toy_interp)

print(f"Verdad:  edad={true_age_myr} Myr, M1={true_M1}, M2={true_M2}")
table_toy

In [ ]:
i_best, j_best = result_toy.best_idx
logAge_best = result_toy.logAge_grid[i_best]
M1_best = result_toy.M1_grid[j_best]
M2_best = result_toy.M2_of_age[i_best]

plot_teff_age_tracks(toy_interp, obs_toy, M1_best);

In [ ]:
plot_radius_age_primary(toy_interp, obs_toy, M1_best);

In [ ]:
plot_radius_mass_secondary(toy_interp, obs_toy, logAge_best, M2_best);

In [ ]:
plot_chi2_map_primary(result_toy);

In [ ]:
plot_chi2_map_secondary(result_toy);

In [ ]:
plot_final_isochrone(toy_interp, obs_toy, logAge_best, M1_best, M2_best);

## 2. Caso real: conecta aquí tus datos

Ajusta rutas y el TIC ID. Esta celda está comentada porque depende de archivos y de conexión a MAST que no están disponibles aquí; descoméntala y ejecútala en tu propio entorno.

In [ ]:
# from astroquery.mast import Catalogs
#
# TIC_ID = "TIC 35159657"
# cat = Catalogs.query_object(TIC_ID, catalog="TIC")
#
# r_A = float(cat["rad"][0])
# teff_A = float(cat["Teff"][0])
# sigma_T = 0.5 * (float(cat["eneg_Teff"][0]) + float(cat["epos_Teff"][0]))
# sigma_r_A = float(cat["e_rad"][0]) if ("e_rad" in cat.colnames and np.isfinite(cat["e_rad"][0])) else 0.1 * r_A
#
# # Radio de la secundaria a partir de la profundidad del tránsito
# # (independiente del modelo evolutivo -- esto sí es correcto usarlo así)
# delta = 0.959
# sigma_delta = 0.05
# R2 = r_A * np.sqrt(delta)
# sigma_R2 = R2 * np.sqrt((sigma_r_A / r_A)**2 + (0.5 * sigma_delta / delta)**2)
#
# obs = BinaryObservables(
#     Teff1=teff_A, sigma_Teff1=sigma_T,
#     R1=r_A, sigma_R1=sigma_r_A,
#     R2=R2, sigma_R2=sigma_R2,
#     # M1=..., sigma_M1=...,   # solo si tienes una masa dinámica independiente del modelo evolutivo
# )
# obs

In [ ]:
# results_by_model = {}
#
# results_by_model["BHAC15"] = run_pipeline(
#     "BHAC15",
#     "/home/luciam/Documents/catalogos_tesis/iso_barafe/BHAC15_iso.GAIA",
#     obs,
#     n_age=250, n_M1=250, n_samples=30000,
# )
#
# results_by_model["PARSEC"] = run_pipeline(
#     "PARSEC",
#     "/home/luciam/Documents/catalogos_tesis/isocronas-parsec/",  # directorio con varios *.dat, uno por edad
#     obs,
#     n_age=250, n_M1=250, n_samples=30000,
# )
#
# main = results_by_model["BHAC15"]
# main["table"]

In [ ]:
# result = main["result"]
# interp = main["interp"]
# i_best, j_best = result.best_idx
# logAge_best = result.logAge_grid[i_best]
# M1_best = result.M1_grid[j_best]
# M2_best = result.M2_of_age[i_best]
#
# plot_teff_age_tracks(interp, obs, M1_best, save_path="teff_vs_edad.png")
# plot_radius_age_primary(interp, obs, M1_best, save_path="radio_vs_edad.png")
# plot_radius_mass_secondary(interp, obs, logAge_best, M2_best, save_path="radio_vs_masa_secundaria.png")
# plot_chi2_map_primary(result, save_path="chi2_M1_edad.png")
# plot_chi2_map_secondary(result, save_path="chi2_M2_edad.png")
# plot_final_isochrone(interp, obs, logAge_best, M1_best, M2_best, save_path="isocrona_final.png")
# plot_model_comparison(results_by_model, save_path="comparacion_modelos.png")
#
# save_results(main["table"], main["samples"], "resultado_binaria")